# ML Provider Benchmark & Architecture Analysis (Legacy 1920x1080 Dataset)
**Heterogeneous 24-CPU + 1-GPU Allocation (CLAIX-2023)**

This notebook contains hard-coded historical results, including separate PhyDLL C++ and Python experiments. For current Score-P validation runs and CUBE-derived breakdowns, use `analysis/current_provider_benchmark.ipynb`.

---

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.autolayout'] = True
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
print('Loaded plotting libraries.')

## 1. Experimental Benchmark Dataset
Data gathered from runs on CLAIX-23 heterogeneous allocation (1x `c23mm` CPU node with 24 ranks, 1x `c23g` GPU node with NVIDIA H100 GPU).
All ML runs evaluated the **`perfect_model`** on a **1920x1080 grid (2,073,600 cells)**.

In [ ]:
data = [
    {
        "Configuration": "No-ML Baseline",
        "Provider": "None",
        "Layout": "N/A",
        "Ranks": "24 CPU",
        "Warmup_Step_ms": 0.0,
        "Steady_Step_ms": 1.289,
        "Total_Mass": 78534900.0,
        "Drift": 0.0,
        "Min_Pos_Water": 0.0117188,
        "Max_Water": 138.129,
        "Moved_Water_Step4": 157259.0,
        "Status": "Passed"
    },
    {
        "Configuration": "SmartSim Direct (Flat)",
        "Provider": "SmartSim Direct",
        "Layout": "flat [N,18]",
        "Ranks": "24 CPU + 1 GPU",
        "Warmup_Step_ms": 363.581,
        "Steady_Step_ms": 104.05,
        "Total_Mass": 78534900.0,
        "Drift": 0.0,
        "Min_Pos_Water": 0.0117188,
        "Max_Water": 138.129,
        "Moved_Water_Step4": 157259.0,
        "Status": "Passed"
    },
    {
        "Configuration": "AIX (25-Rank 5x5 MPMD)",
        "Provider": "AIX",
        "Layout": "flat [N,18]",
        "Ranks": "24 CPU + 1 GPU (25 total)",
        "Warmup_Step_ms": 2261.19,
        "Steady_Step_ms": 117.158,
        "Total_Mass": 78534900.0,
        "Drift": 0.0,
        "Min_Pos_Water": 0.0117188,
        "Max_Water": 138.129,
        "Moved_Water_Step4": 157259.0,
        "Status": "Passed"
    },
    {
        "Configuration": "SmartSim CMI (Flat)",
        "Provider": "SmartSim CMI",
        "Layout": "flat [N,18]",
        "Ranks": "24 CPU + 1 GPU",
        "Warmup_Step_ms": 288.019,
        "Steady_Step_ms": 146.976,
        "Total_Mass": 78534900.0,
        "Drift": 0.0,
        "Min_Pos_Water": 0.0117188,
        "Max_Water": 138.129,
        "Moved_Water_Step4": 157259.0,
        "Status": "Passed"
    },
    {
        "Configuration": "PhyDLL C++",
        "Provider": "PhyDLL C++",
        "Layout": "flat [N,18]",
        "Ranks": "24 CPU + 1 GPU",
        "Warmup_Step_ms": 6492.0,
        "Steady_Step_ms": 763.299,
        "Total_Mass": 78534900.0,
        "Drift": 0.0,
        "Min_Pos_Water": 0.0117188,
        "Max_Water": 138.129,
        "Moved_Water_Step4": 157259.0,
        "Status": "Passed"
    },
    {
        "Configuration": "PhyDLL Python",
        "Provider": "PhyDLL Python",
        "Layout": "flat [N,18]",
        "Ranks": "24 CPU + 1 GPU",
        "Warmup_Step_ms": 3088.73,
        "Steady_Step_ms": 1880.95,
        "Total_Mass": 78534900.0,
        "Drift": 0.0,
        "Min_Pos_Water": 0.0117188,
        "Max_Water": 138.129,
        "Moved_Water_Step4": 157259.0,
        "Status": "Passed"
    }
]
df = pd.DataFrame(data)
df

## 2. Warmup vs Steady-State Inference Performance
Comparison of Step 2 (Initial ML step with model loading & CUDA warm-up) vs Step 4 (Steady-state ML step).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ml_df = df[df['Provider'] != 'None'].sort_values('Steady_Step_ms')

x = np.arange(len(ml_df))
width = 0.35

rects1 = ax.bar(x - width/2, ml_df['Warmup_Step_ms'], width, label='Step 2 (Warmup / Initial Load)', color='#dd8452', edgecolor='black')
rects2 = ax.bar(x + width/2, ml_df['Steady_Step_ms'], width, label='Step 4 (Steady-State Inference)', color='#4c72b0', edgecolor='black')

ax.set_ylabel('Execution Time (ms, log scale)', fontweight='bold')
ax.set_title('Warmup vs Steady-State ML Step Execution Time (1920x1080 Grid, Perfect Model)', fontweight='bold', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(ml_df['Configuration'])
ax.set_yscale('log')
ax.set_ylim(50, 12000)
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.legend(frameon=True)

for rect in rects2:
    height = rect.get_height()
    ax.annotate(f'{height:.1f} ms',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontweight='bold', fontsize=9)

plt.savefig('fig1_steady_state_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Physical Correctness & Mass Conservation Verification
Comparison of liquid mass, water movement, and min/max water values against the **No-ML Ground Truth Baseline**.

In [ ]:
corr_df = df[['Configuration', 'Total_Mass', 'Drift', 'Min_Pos_Water', 'Max_Water', 'Moved_Water_Step4', 'Status']].copy()
corr_df['Mass_Match'] = corr_df['Total_Mass'].apply(lambda x: '✅ Match' if abs(x - 7.85349e7) < 1e2 else '❌ Discrepancy')
corr_df

## 4. Summary & Key Findings
1. **Steady-State Equivalence**: In steady state (Step 4), **SmartSim Direct** (`104.05 ms`), **AIX 25-Rank MPMD** (`117.16 ms`), and **SmartSim CMI** (`146.98 ms`) achieve virtually **identical ~100–150ms execution latencies** for 2.07 million grid cells.
2. **Warmup Overhead**: The initial gap in Step 2 was driven by **model loading, JIT compilation, and CUDA context initialization**.
3. **PhyDLL Bottleneck**: PhyDLL sequential metadata/field exchanges limit steady-state latency to ~0.76s (C++) / ~1.88s (Python).